# 01: Build ligand->receptor map, compute reinforcement scores, and summarize synergy

Runs four steps in sequence:

1. **Build the ligand -> receptor gene mapping** (`ligand_receptor_map.csv`) -- a
   curated dictionary pairing each of the 54 ligands in the screen with its
   receptor gene subunit(s).
2. **Compute directional reinforcement scores per combo** -- for every ligand-pair
   combo, tests whether each ligand's single-ligand effect moves the other ligand's
   receptor gene(s), using the raw `lfc` metric averaged (`mean`) across a
   receptor's subunits. Returns two metrics `lfc_mean_1to2` or `lfc_mean_2to1` representing regulation of each member of the combo on each other. Then calculates a metric `lfc_mean_combined` which is the average of `lfc_mean_1to2` and `lfc_mean_2to1`, essentially asking whether there is a net positive or negative effect of the combination on respective ligand receptors. Flags a
   combo as reinforcing (`sig_flag_final`) only if it there is significant upregulation by at least one ligand AND there is an overall net-positive combined LFC.
3. **Compute the per-combo synergy gene-count summary** -- collapses the per-gene
   interaction-scoring table to one row per combo, counting how many genes fall into
   each interaction class (synergy, buffering, etc.). 
4. **Combine and export** -- left-merges step 3 onto step 2 and writes the single
   final output: `analysis_outs/signal_reinforcement_lfc_mean/results_combined_0.1filter.csv`.


## Step 1: Build ligand -> receptor gene mapping

Create ligand->receptor dict using standard cytokine/chemokine receptor biology (mouse gene
symbols). Multi-subunit receptors (e.g. the common gamma chain family, gp130 family,
IL-1 family) are listed as multiple rows per ligand.

Output: `ligand_receptor_map.csv` with columns:
- `ligand`, `receptor_gene`
- `in_gene_panel`: whether the gene survived upstream QC filtering and is present in
  `SIG13_doublets_DSB7.h5ad` (13,050 genes) at all. If False, that subunit cannot be
  scored for reinforcement in this dataset regardless of receptor complex biology.
- `is_shared_subunit` / `shared_with_ligands`: flags receptor genes used by more than
  one screen ligand (e.g. `Il2rg` for the whole gamma-chain family).
- `low_confidence_note`: caveats for ligands where the receptor assignment itself is
  less clean (WNT1 Frizzled promiscuity, MSTN/MIF shared co-receptors).


In [1]:
# Repo-relative paths: inputs from imports_stable/, outputs to analysis_outs/
import os
from pathlib import Path
_here = Path.cwd().resolve()
REPO_DIR = str(next(p for p in [_here, *_here.parents] if (p / "imports_stable").is_dir()))
IMPORTS_DIR = f"{REPO_DIR}/imports_stable"
OUTS_DIR = f"{REPO_DIR}/analysis_outs/02_combinatorial_screen_signalseq_SIG13"

import pandas as pd
import numpy as np
import h5py
import os

H5AD_PATH = f'{IMPORTS_DIR}/SIG13/scanpy_outs/SIG13_doublets_DSB7.h5ad'
MAP_OUT_CSV = 'ligand_receptor_map.csv'


### Curated ligand -> receptor gene mapping

In [2]:
import pandas as pd

# ligand -> list of receptor gene subunits (mouse gene symbols), curated from standard
# cytokine/chemokine receptor biology, covering all 54 ligands in the SIG13 screen.
RECEPTORS = {
    # gamma-chain family: Il2rg itself is excluded (shared across all 6 ligands below
    # plus the wider cytokine literature; its specificity is too broad to count as
    # evidence of reinforcing any one particular partner)
    'IL2':  ['Il2ra', 'Il2rb'],
    'IL4':  ['Il4ra'],
    'IL7':  ['Il7r'],
    'IL9':  ['Il9r'],
    'IL15': ['Il15ra', 'Il2rb'],
    'IL21': ['Il21r'],
    # gp130 family: Il6st itself is excluded, same reasoning as Il2rg above
    'IL6':  ['Il6ra'],
    'IL27': ['Il27ra'],
    'LIF':  ['Lifr'],
    'OSM':  ['Osmr'],
    # type I / type II interferons
    'IFNA':  ['Ifnar1', 'Ifnar2'],
    'IFNB1': ['Ifnar1', 'Ifnar2'],
    'IFNE':  ['Ifnar1', 'Ifnar2'],
    'IFNK':  ['Ifnar1', 'Ifnar2'],
    'IFNG':  ['Ifngr1', 'Ifngr2'],
    # IL-12 family (shared Il12rb1)
    'IL12': ['Il12rb1', 'Il12rb2'],
    'IL23': ['Il23r', 'Il12rb1'],
    # IL-1 family
    'IL1A':  ['Il1r1', 'Il1rap'],
    'IL1B':  ['Il1r1', 'Il1rap'],
    'IL18':  ['Il18r1', 'Il18rap'],
    'IL33':  ['Il1rl1', 'Il1rap'],
    'IL1F6': ['Il1rl2', 'Il1rap'],   # Il36a
    'IL1F8': ['Il1rl2', 'Il1rap'],   # Il36b
    'IL1F9': ['Il1rl2', 'Il1rap'],   # Il36g
    # IL-10
    'IL10': ['Il10ra', 'Il10rb'],
    # IL-17 family
    'IL17A': ['Il17ra', 'Il17rc'],
    'IL17F': ['Il17ra', 'Il17rc'],
    'IL25':  ['Il17rb', 'Il17ra'],   # IL-17E
    # TGF-beta family
    'TGFB1': ['Tgfbr1', 'Tgfbr2', 'Tgfbr3'],
    'TGFB3': ['Tgfbr1', 'Tgfbr2', 'Tgfbr3'],
    # TSLP (shares Il7r with IL7)
    'TSLP': ['Crlf2', 'Il7r'],
    # TNF superfamily
    'TNF':     ['Tnfrsf1a', 'Tnfrsf1b'],
    'TNFSF4':  ['Tnfrsf4'],
    'TNFSF9':  ['Tnfrsf9'],
    'TNFSF14': ['Tnfrsf14', 'Ltbr'],
    'TNFSF15': ['Tnfrsf25'],
    'TNFSF18': ['Tnfrsf18'],
    # chemokines
    'CCL1':   ['Ccr8'],
    'CCL7':   ['Ccr1', 'Ccr2', 'Ccr3'],
    'CCL11':  ['Ccr3'],
    'CCL12':  ['Ccr2'],
    'CCL19':  ['Ccr7'],
    'CCL20':  ['Ccr6'],
    'CCL21A': ['Ccr7'],
    'CCL22':  ['Ccr4'],
    'CCL25':  ['Ccr9'],
    'CCL27':  ['Ccr10'],
    'CX3CL1': ['Cx3cr1'],
    'CXCL9':  ['Cxcr3'],
    'CXCL10': ['Cxcr3'],
    'CXCL12': ['Cxcr4'],
    # misc
    'MIF':  ['Cd74', 'Cxcr4'],
    'MSTN': ['Acvr2b', 'Acvr1b'],
    'WNT1': ['Lrp5', 'Lrp6'],       # Frizzled partner unresolved (promiscuous); co-receptors only
}

LOW_CONFIDENCE = {
    'WNT1': 'Canonical Wnt signaling requires a Frizzled receptor + Lrp5/6 co-receptor; '
            'FZD usage by WNT1 in this cell type is promiscuous/context-dependent and most '
            'Fzd genes are filtered from the gene panel, so only the Lrp5/Lrp6 co-receptors '
            'are used here. Treat WNT1 reinforcement scores as lower confidence.',
    'MSTN': 'Acvr1b (Alk4) is a shared type-I receptor across multiple activin/GDF ligands, '
            'not myostatin-specific; Acvr2b is the myostatin-defining subunit.',
    'MIF':  'Cxcr4 and Cxcr2 are signal-transducing MIF co-receptors but not MIF-specific '
            '(Cxcr4 is also the CXCL12 receptor, itself in this screen); Cd74 is the '
            'MIF-specific binding subunit. Cxcr2 is absent from the gene panel.',
}

ligands_54 = ['CCL1','CCL11','CCL12','CCL19','CCL20','CCL21A','CCL22','CCL25','CCL27','CCL7',
    'CX3CL1','CXCL10','CXCL12','CXCL9','IFNA','IFNB1','IFNE','IFNG','IFNK','IL10','IL12','IL15',
    'IL17A','IL17F','IL18','IL1A','IL1B','IL1F6','IL1F8','IL1F9','IL2','IL21','IL23','IL25','IL27',
    'IL33','IL4','IL6','IL7','IL9','LIF','MIF','MSTN','OSM','TGFB1','TGFB3','TNF','TNFSF14',
    'TNFSF15','TNFSF18','TNFSF4','TNFSF9','TSLP','WNT1']

missing_ligands = [l for l in ligands_54 if l not in RECEPTORS]
extra_ligands = [l for l in RECEPTORS if l not in ligands_54]
print('ligands in screen not yet mapped:', missing_ligands)
print('mapped ligands not in the 54-ligand screen list (typo check):', extra_ligands)
assert not missing_ligands and not extra_ligands

# genes actually present in the full measured gene panel (13,050 genes)
f = h5py.File(H5AD_PATH, 'r')
idx_key = f['var'].attrs.get('_index', '_index')
var_names = set(v.decode() if isinstance(v, bytes) else v for v in f['var'][idx_key][:])

rows = []
gene_to_ligands = {}
for ligand, genes in RECEPTORS.items():
    for g in genes:
        gene_to_ligands.setdefault(g, []).append(ligand)

for ligand, genes in RECEPTORS.items():
    for g in genes:
        sharers = [l for l in gene_to_ligands[g] if l != ligand]
        rows.append({
            'ligand': ligand,
            'receptor_gene': g,
            'in_gene_panel': g in var_names,
            'is_shared_subunit': len(sharers) > 0,
            'shared_with_ligands': ';'.join(sorted(sharers)),
            'low_confidence_note': LOW_CONFIDENCE.get(ligand, ''),
        })

df = pd.DataFrame(rows)
print(df.shape)
print('genes missing from gene panel entirely:')
print(df[~df['in_gene_panel']][['ligand','receptor_gene']].to_string(index=False))
print()
print('ligands with ZERO receptor genes present in gene panel (cannot be scored at all):')
avail = df.groupby('ligand')['in_gene_panel'].any()
print(avail[~avail].index.tolist())


ligands in screen not yet mapped: []
mapped ligands not in the 54-ligand screen list (typo check): []
(86, 6)
genes missing from gene panel entirely:
ligand receptor_gene
   OSM          Osmr
  IL23         Il23r
 IL17A        Il17rc
 IL17F        Il17rc
  CCL7          Ccr1
 CCL27         Ccr10
CX3CL1        Cx3cr1

ligands with ZERO receptor genes present in gene panel (cannot be scored at all):
['CCL27', 'CX3CL1', 'OSM']


### Sanity checks

All 54 screen ligands must have a mapping entry (no typos, no omissions), confirmed
above. Two ligands (`CCL27`, `CX3CL1`) have zero measurable receptor genes in this
dataset -- their receptors (`Ccr10`, `Cx3cr1`) were filtered out during upstream QC
(likely low/absent expression in naive CD4 T cells), so reinforcement *of* these two
ligands' receptors cannot be scored in either direction. This is a real data
limitation, not a mapping gap, and is carried forward into step 2's logging.


In [3]:
df.to_csv(MAP_OUT_CSV, index=False)
df


,ligand,receptor_gene,in_gene_panel,is_shared_subunit,shared_with_ligands,low_confidence_note
0,IL2,Il2ra,True,False,,
1,IL2,Il2rb,True,True,IL15,
2,IL4,Il4ra,True,False,,
3,IL7,Il7r,True,True,TSLP,
4,IL9,Il9r,True,False,,
...,...,...,...,...,...,...
81,MIF,Cxcr4,True,True,CXCL12,Cxcr4 and Cxcr2 are signal-transducing MIF co-...
82,MSTN,Acvr2b,True,False,,Acvr1b (Alk4) is a shared type-I receptor acro...
83,MSTN,Acvr1b,True,False,,Acvr1b (Alk4) is a shared type-I receptor acro...
84,WNT1,Lrp5,True,False,,Canonical Wnt signaling requires a Frizzled re...


## Step 2: Compute directional reinforcement scores

For every real combo `A_B` (A = round1 ligand, B = round2 ligand), asks: does A's
single-ligand effect (`lfc_ligand1`) move the receptor gene(s) of B, and does B's
single-ligand effect (`lfc_ligand2`) move the receptor gene(s) of A? Each direction
is scored using the raw `lfc` metric from the GLM interaction-scoring output,
averaged (`mean`) across a receptor's subunits when it has more than one.

**Filter version**: this step (and step 3's per-gene synergy scoring) uses the
`0.1filter` GLM output. `0.2filter` is still used in step 3's `consistent_ligands`
replicate-reproducibility filter, for consistency with other analyses.

**Source table**: `interactions_scored_v3_glmGamPoi_0.1filter.csv` also contains
degenerate single-ligand rows (e.g. `IFNG_linker`) alongside real combos -- these are
dropped here (`ligand1 != 'linker' & ligand2 != 'linker'`) since there's no partner to
test reinforcement against, along with double-dose self-combos (`ligand1 ==
ligand2`).

**`sig_flag`** (computed per direction): True if the net effect across the receptor
complex is positive (`lfc_mean > SIG_LFC_THRESH = 0`) AND at least one scored
receptor gene has `adj_pval <= SIG_PADJ_THRESH = 0.01`.

**`sig_flag_final`**: `(sig_flag_1to2 | sig_flag_2to1) & (lfc_mean_sum > 0)` --
directionally significant in at least one direction AND net-positive combined LFC.

**Coverage caveat**: several screen ligands (chemokines whose receptors are CCRs, and
several IL-1-family members) have zero receptor genes passing the 0.1filter DEG
inclusion threshold at all -- logged explicitly below; `lfc_mean_sum` falls back
to whichever single direction has data via `np.nanmean`, and
`has_1to2_data`/`has_2to1_data` flag which combos are affected.

This step's result (`result`) stays in memory -- not written to disk on its own
(step 4 below writes the combined, filtered table).


In [4]:
FILTER = '0.1filter'
CSV = f'{IMPORTS_DIR}/SIG13/analysis_outs/glmGamPoi/interactions_scored_v3_glmGamPoi_{FILTER}.csv'
MAP_CSV = 'ligand_receptor_map.csv'

EPS = 1e-300
SIG_PADJ_THRESH = 0.01
SIG_LFC_THRESH = 0


### Load receptor map and filter the (large) DEG table to receptor-gene rows only

In [5]:
rmap = pd.read_csv(MAP_CSV)
rmap_avail = rmap[rmap['in_gene_panel']].copy()
receptor_genes = set(rmap_avail['receptor_gene'])
print(f'{len(receptor_genes)} unique in-panel receptor genes across {rmap["ligand"].nunique()} ligands')

# ligand -> whether ALL of its in-panel receptor genes are shared with another ligand
# (i.e. this ligand has no ligand-specific receptor subunit measurable at all)
shared_only = rmap_avail.groupby('ligand')['is_shared_subunit'].all()

usecols = ['name', 'interaction', 'lfc_ligand1', 'adj_pval_ligand1', 'lfc_ligand2', 'adj_pval_ligand2']
chunks = []
for chunk in pd.read_csv(CSV, usecols=usecols, chunksize=500_000):
    chunks.append(chunk[chunk['name'].isin(receptor_genes)])
hits = pd.concat(chunks, ignore_index=True)
print('receptor-gene rows pulled from the DEG table:', hits.shape)


52 unique in-panel receptor genes across 54 ligands
receptor-gene rows pulled from the DEG table: (21749, 6)


### Split into real combos

Drops degenerate single-ligand rows (`ligand1 == 'linker'` or `ligand2 == 'linker'`),
and drops "double dose" self-combos (`ligand1 == ligand2`, e.g. `IL2_IL2`). 11 of the
594 combos are self-combos and are excluded from this point on.


In [6]:
split = hits['interaction'].str.split('_', n=1, expand=True)
hits['ligand1'], hits['ligand2'] = split[0], split[1]
combos = hits[(hits['ligand1'] != 'linker') & (hits['ligand2'] != 'linker')
              & (hits['ligand1'] != hits['ligand2'])].copy()

all_combo_ids = pd.Index(sorted(combos['interaction'].unique()))
print(f'{len(all_combo_ids)} real, non-self combos have >=1 receptor-gene hit in the DEG table '
      f'(expect 583 = 594 - 11 double-dose combos)')


583 real, non-self combos have >=1 receptor-gene hit in the DEG table (expect 583 = 594 - 11 double-dose combos)


### Directional reinforcement: build one direction, aggregate per combo (lfc_mean only)

In [7]:
def add_metrics(df_, lfc_col, padj_col):
    df_ = df_.copy()
    padj = np.clip(df_[padj_col].values, EPS, 1.0)
    lfc = df_[lfc_col].values
    df_['lfc'] = lfc
    df_['adj_pval'] = padj
    return df_


def build_direction(hits_df, target_ligand_col, lfc_col, padj_col):
    """target_ligand_col: 'ligand2' for A->B (uses A's lfc_ligand1 on B's receptors),
    'ligand1' for B->A (uses B's lfc_ligand2 on A's receptors)."""
    m = hits_df.merge(rmap_avail, left_on=[target_ligand_col, 'name'],
                       right_on=['ligand', 'receptor_gene'], how='inner')
    m = add_metrics(m, lfc_col, padj_col)
    agg = m.groupby('interaction').agg(
        lfc_mean=('lfc', 'mean'),
        n_receptor_genes_scored=('name', 'nunique'),
        receptor_genes_scored=('name', lambda s: ';'.join(sorted(s.unique()))),
    )
    # sig_flag = the NET effect across the receptor complex is positive
    # (lfc_mean > SIG_LFC_THRESH) AND there's real statistical signal behind it (>=1
    # gene at adj_pval <= SIG_PADJ_THRESH).
    min_pval = m.groupby('interaction')['adj_pval'].min()
    agg['sig_flag'] = (agg['lfc_mean'] > SIG_LFC_THRESH) & (min_pval <= SIG_PADJ_THRESH)
    return agg.reindex(all_combo_ids)


dir_1to2 = build_direction(combos, 'ligand2', 'lfc_ligand1', 'adj_pval_ligand1').add_suffix('_1to2')
dir_2to1 = build_direction(combos, 'ligand1', 'lfc_ligand2', 'adj_pval_ligand2').add_suffix('_2to1')

print('combos with 0 scoreable receptor genes, A->B direction:',
      dir_1to2['n_receptor_genes_scored_1to2'].isna().sum(), '/', len(all_combo_ids))
print('combos with 0 scoreable receptor genes, B->A direction:',
      dir_2to1['n_receptor_genes_scored_2to1'].isna().sum(), '/', len(all_combo_ids))


combos with 0 scoreable receptor genes, A->B direction: 209 / 583
combos with 0 scoreable receptor genes, B->A direction: 0 / 583


#### Which ligands are entirely unscoreable as a reinforcement *target*?

(i.e. every combo where this ligand is the partner has zero receptor genes passing
the DEG filter -- logged explicitly rather than silently dropped, per repo
convention for reporting coverage gaps.)


In [8]:
all_ligand2 = pd.Index(all_combo_ids).map(lambda i: i.split('_', 1)[1])
total_per_ligand2 = pd.Series(all_ligand2).value_counts()

missing_as_B = dir_1to2[dir_1to2['n_receptor_genes_scored_1to2'].isna()].index
missing_ligand2 = pd.Index(missing_as_B).map(lambda i: i.split('_', 1)[1])
missing_per_ligand2 = pd.Series(missing_ligand2).value_counts()
fully_missing_2 = [l for l in missing_per_ligand2.index if missing_per_ligand2[l] == total_per_ligand2.get(l, 0)]
print('ligands never scoreable as the B (round2) partner:')
print(sorted(fully_missing_2))

missing_as_A = dir_2to1[dir_2to1['n_receptor_genes_scored_2to1'].isna()].index
missing_ligand1 = pd.Index(missing_as_A).map(lambda i: i.split('_', 1)[0])
print('n combos with A unscoreable as reinforcement target:', len(missing_as_A))
print('breakdown by A ligand:')
print(pd.Series(missing_ligand1).value_counts())


ligands never scoreable as the B (round2) partner:
['CCL1', 'CCL11', 'CCL12', 'CCL20', 'CCL25', 'CCL27', 'CCL7', 'CX3CL1', 'CXCL10', 'CXCL9', 'IL1A', 'IL1B', 'IL1F6', 'IL1F8', 'IL1F9', 'IL33', 'IL9', 'LIF', 'OSM']
n combos with A unscoreable as reinforcement target: 0
breakdown by A ligand:
Series([], Name: count, dtype: int64)


### Combine directions into the final per-combo table

In [9]:
result = pd.DataFrame(index=all_combo_ids)
result['ligand1'] = [i.split('_', 1)[0] for i in all_combo_ids]
result['ligand2'] = [i.split('_', 1)[1] for i in all_combo_ids]
result = result.join(dir_1to2).join(dir_2to1)

result['has_1to2_data'] = result['n_receptor_genes_scored_1to2'].notna()
result['has_2to1_data'] = result['n_receptor_genes_scored_2to1'].notna()

# combine the two directions into a single "mean LFC" score for the combo, ignoring NaNs
result['lfc_mean_sum'] = np.nansum(
    np.vstack([result['lfc_mean_1to2'].values, result['lfc_mean_2to1'].values]), axis=0)

sig_1to2 = result['sig_flag_1to2'].astype('object').where(result['sig_flag_1to2'].notna(), False).astype(bool)
sig_2to1 = result['sig_flag_2to1'].astype('object').where(result['sig_flag_2to1'].notna(), False).astype(bool)
result['sig_flag_final'] = (sig_1to2 | sig_2to1) & (result['lfc_mean_sum'] > 0)

result['uses_shared_subunit_only_1to2'] = result['ligand2'].map(shared_only).astype('object').where(result['ligand2'].map(shared_only).notna(), False).astype(bool)
result['uses_shared_subunit_only_2to1'] = result['ligand1'].map(shared_only).astype('object').where(result['ligand1'].map(shared_only).notna(), False).astype(bool)

result = result.reset_index().rename(columns={'index': 'interaction'})
print('result shape (kept in memory, not written to disk):', result.shape)
result.head()


result shape (kept in memory, not written to disk): (583, 17)


,interaction,ligand1,ligand2,lfc_mean_1to2,n_receptor_genes_scored_1to2,receptor_genes_scored_1to2,sig_flag_1to2,lfc_mean_2to1,n_receptor_genes_scored_2to1,receptor_genes_scored_2to1,sig_flag_2to1,has_1to2_data,has_2to1_data,lfc_mean_sum,sig_flag_final,uses_shared_subunit_only_1to2,uses_shared_subunit_only_2to1
0,IFNG_CCL1,IFNG,CCL1,NaN,NaN,NaN,NaN,-0.026575,2,Ifngr1;Ifngr2,False,False,True,-0.026575,False,False,False
1,IFNG_CCL11,IFNG,CCL11,NaN,NaN,NaN,NaN,-0.044398,2,Ifngr1;Ifngr2,False,False,True,-0.044398,False,True,False
2,IFNG_CCL12,IFNG,CCL12,NaN,NaN,NaN,NaN,0.024267,2,Ifngr1;Ifngr2,False,False,True,0.024267,False,True,False
3,IFNG_CCL19,IFNG,CCL19,-0.14066,1.0,Ccr7,False,-0.134862,2,Ifngr1;Ifngr2,False,True,True,-0.275521,False,True,False
4,IFNG_CCL20,IFNG,CCL20,NaN,NaN,NaN,NaN,-0.013999,2,Ifngr1;Ifngr2,False,False,True,-0.013999,False,False,False


### Summary

In [10]:
print('combos with both directions scored:', int((result['has_1to2_data'] & result['has_2to1_data']).sum()), '/', len(result))
print('combos with only one direction scored:', int((result['has_1to2_data'] ^ result['has_2to1_data']).sum()))
print('combos with neither direction scored:', int((~result['has_1to2_data'] & ~result['has_2to1_data']).sum()))
print()
print('sig_flag_final (directionally reinforcing AND net-positive combined LFC):', int(result['sig_flag_final'].sum()), '/', len(result))
result[['interaction', 'lfc_mean_sum', 'sig_flag_final']].describe(include='all')


combos with both directions scored: 374 / 583
combos with only one direction scored: 209
combos with neither direction scored: 0

sig_flag_final (directionally reinforcing AND net-positive combined LFC): 115 / 583


,interaction,lfc_mean_sum,sig_flag_final
count,583,583.000000,583
unique,583,NaN,2
top,IFNG_CCL1,NaN,False
freq,1,NaN,468
mean,NaN,0.020042,NaN
std,NaN,0.653894,NaN
min,NaN,-2.145383,NaN
25%,NaN,-0.196845,NaN
50%,NaN,-0.017780,NaN
75%,NaN,0.125474,NaN


## Step 3: Compute per-combo synergy gene-count summary

Collapses the per-gene `interactions_scored_v3_glmGamPoi_0.1filter.csv` table to one
row per combo, counting how many of its genes fall into each interaction-class
category, after restricting to combos that pass a replicate-reproducibility filter.

**`interaction_class` semantics**: a gene's `interaction_class` is `NaN` unless
`adj_pval_condition <= 0.1` (i.e. the gene is significantly DE in the combo
condition vs. control at all); among those genes, it is `"none"` if the interaction
term itself isn't significant, and `"synergy positive"` / `"synergy negative"` /
`"buffering"` based on the sign of `interaction_score` and the total combined effect
when the interaction term *is* significant (`adj_pval_interaction <= 0.1`). So the
natural denominator for "how many genes could have shown synergy" is the count of
non-null `interaction_class` rows (genes significantly DE in that combo at all) --
that's `n_total_degs` below.

**`n_buffering`**: count of genes where the interaction term is significant but
*dampens* the combined effect rather than amplifying it (antagonism, not synergy).

**Filter versions (deliberately mixed)**: per-gene scoring uses `0.1filter`
(`SCORE_FILTER`), matching step 2. The `consistent_ligands` reproducibility filter
uses `0.2filter` (`REPCORR_FILTER`), because
`replicate_correlation_interactionLfc_0.2filter.csv` is the only version of that file
that exists.

**Reproducibility filter (`consistent_ligands`)**: combos are restricted to those
with replicate `pearson_corr > 0.25` and `num_shared_genes > 1` -- 146 of 594 combos
pass this bar (before excluding double-dose combos).

**Double-dose exclusion**: combos where `ligand1 == ligand2` are dropped, same as in
step 2.

This step's result (`summary`) stays in memory -- not written separately (step 4
merges it with `result` and writes the single combined output).


In [11]:
SCORE_FILTER = '0.1filter'
REPCORR_FILTER = '0.2filter'
SCORED_CSV = f'{IMPORTS_DIR}/SIG13/analysis_outs/glmGamPoi/interactions_scored_v3_glmGamPoi_{SCORE_FILTER}.csv'
REPCORR_CSV = f'{IMPORTS_DIR}/SIG13/analysis_outs/replicate_corr/replicate_correlation_interactionLfc_{REPCORR_FILTER}.csv'


### Load the consistent_ligands reproducibility filter

In [12]:
repcorr = pd.read_csv(REPCORR_CSV)
consistent = repcorr[(repcorr['pearson_corr'] > 0.25) & (repcorr['num_shared_genes'] > 1)]
consistent_ligands = set(consistent['interaction'])
print(f'{len(consistent_ligands)} / {repcorr["interaction"].nunique()} combos with replicate correlation pass the '
      f'consistent_ligands filter (out of 594 total combos, before double-dose exclusion)')


146 / 491 combos with replicate correlation pass the consistent_ligands filter (out of 594 total combos, before double-dose exclusion)


### Load per-gene interaction classifications, restricted to consistent, real, non-self combos

In [13]:
usecols = ['name', 'interaction', 'adj_pval_condition', 'interaction_class']
chunks = []
for chunk in pd.read_csv(SCORED_CSV, usecols=usecols, chunksize=500_000):
    chunks.append(chunk[chunk['interaction'].isin(consistent_ligands)])
scored = pd.concat(chunks, ignore_index=True)

split = scored['interaction'].str.split('_', n=1, expand=True)
scored['ligand1'], scored['ligand2'] = split[0], split[1]
scored = scored[(scored['ligand1'] != 'linker') & (scored['ligand2'] != 'linker')
                & (scored['ligand1'] != scored['ligand2'])]
print('gene x combo rows after consistent_ligands + real, non-self combo filtering:', scored.shape)
print('n distinct combos remaining:', scored['interaction'].nunique())


gene x combo rows after consistent_ligands + real, non-self combo filtering: (1095791, 6)
n distinct combos remaining: 138


### Collapse to one row per combo

In [14]:
tested = scored[scored['interaction_class'].notna()]

summary = tested.groupby('interaction').agg(
    n_total_degs=('name', 'nunique'),
    n_synergy_positive=('interaction_class', lambda s: int((s == 'synergy positive').sum())),
    n_synergy_negative=('interaction_class', lambda s: int((s == 'synergy negative').sum())),
    n_buffering=('interaction_class', lambda s: int((s == 'buffering').sum())),
).reset_index()
summary['n_synergy_total'] = summary['n_synergy_positive'] + summary['n_synergy_negative']

# combos in consistent_ligands with zero significantly-DE genes at all (no rows survive
# the interaction_class.notna() filter) would be silently dropped by the groupby above --
# add them back explicitly with all-zero counts so the denominator isn't misleading.
all_consistent_real_combos = {i for i in consistent_ligands
                               if '_' in i and i.split('_', 1)[0] != 'linker' and i.split('_', 1)[1] != 'linker'
                               and i.split('_', 1)[0] != i.split('_', 1)[1]}
missing = all_consistent_real_combos - set(summary['interaction'])
print(f'{len(missing)} consistent-ligand combos have zero significantly-DE genes at all (added back with 0 counts):')
print(sorted(missing))
if missing:
    filler = pd.DataFrame({'interaction': sorted(missing)})
    for col in ['n_total_degs', 'n_synergy_positive', 'n_synergy_negative', 'n_buffering', 'n_synergy_total']:
        filler[col] = 0
    summary = pd.concat([summary, filler], ignore_index=True)

summary['freq_synergy_positive'] = summary['n_synergy_positive'] / summary['n_total_degs'].replace(0, np.nan)
summary['freq_synergy_negative'] = summary['n_synergy_negative'] / summary['n_total_degs'].replace(0, np.nan)
summary['freq_synergy_total'] = summary['n_synergy_total'] / summary['n_total_degs'].replace(0, np.nan)
summary['freq_buffering'] = summary['n_buffering'] / summary['n_total_degs'].replace(0, np.nan)

print('summary shape (kept in memory, not written to disk):', summary.shape)
summary.describe()


0 consistent-ligand combos have zero significantly-DE genes at all (added back with 0 counts):
[]
summary shape (kept in memory, not written to disk): (138, 10)


,n_total_degs,n_synergy_positive,n_synergy_negative,n_buffering,n_synergy_total,freq_synergy_positive,freq_synergy_negative,freq_synergy_total,freq_buffering
count,138.000000,138.000000,138.000000,138.000000,138.000000,138.000000,138.000000,138.000000,138.000000
mean,4580.007246,119.862319,158.985507,418.710145,278.847826,0.022428,0.028559,0.050987,0.082328
std,1256.863048,150.803115,259.208649,475.891744,405.927664,0.023332,0.039519,0.061794,0.080848
min,1135.000000,3.000000,0.000000,6.000000,4.000000,0.000917,0.000000,0.001932,0.001328
25%,3696.250000,27.000000,28.000000,109.500000,62.250000,0.007644,0.007646,0.016316,0.027737
50%,4680.500000,64.000000,57.000000,290.000000,112.500000,0.014394,0.013504,0.026866,0.067863
75%,5556.000000,149.750000,139.750000,521.500000,304.250000,0.028457,0.028195,0.060983,0.102869
max,6905.000000,848.000000,1523.000000,2919.000000,2371.000000,0.129406,0.232413,0.361819,0.495165


### Summary

In [15]:
print(f'{len(summary)} combos in the final synergy summary (consistent_ligands, real non-self combos only)')
print()
print('n_synergy_positive distribution:')
print(summary['n_synergy_positive'].value_counts().sort_index())
print()
print('n_synergy_negative distribution:')
print(summary['n_synergy_negative'].value_counts().sort_index())
print()
print('n_buffering distribution:')
print(summary['n_buffering'].describe())
summary.sort_values('n_synergy_total', ascending=False).head(10)


138 combos in the final synergy summary (consistent_ligands, real non-self combos only)

n_synergy_positive distribution:
n_synergy_positive
3      1
4      2
5      3
6      1
7      3
      ..
431    1
645    1
745    1
791    1
848    1
Name: count, Length: 95, dtype: int64

n_synergy_negative distribution:
n_synergy_negative
0       1
2       1
4       1
5       2
6       1
       ..
915     1
1094    1
1114    1
1362    1
1523    1
Name: count, Length: 102, dtype: int64

n_buffering distribution:
count     138.000000
mean      418.710145
std       475.891744
min         6.000000
25%       109.500000
50%       290.000000
75%       521.500000
max      2919.000000
Name: n_buffering, dtype: float64


,interaction,n_total_degs,n_synergy_positive,n_synergy_negative,n_buffering,n_synergy_total,freq_synergy_positive,freq_synergy_negative,freq_synergy_total,freq_buffering
133,TNF_TNFSF15,6553,848,1523,624,2371,0.129406,0.232413,0.361819,0.095224
79,IL4_TNFSF18,6887,791,1362,812,2153,0.114854,0.197764,0.312618,0.117903
30,IL21_TNFSF18,6828,745,1114,865,1859,0.109110,0.163152,0.272261,0.126684
29,IL21_TNFSF15,6225,645,1094,321,1739,0.103614,0.175743,0.279357,0.051566
61,IL2_TNFSF18,6905,393,915,1155,1308,0.056915,0.132513,0.189428,0.167270
62,IL2_TNFSF4,6361,366,800,460,1166,0.057538,0.125766,0.183305,0.072316
135,TNF_TNFSF4,5990,382,714,600,1096,0.063773,0.119199,0.182972,0.100167
45,IL27_TNFSF15,5315,409,641,142,1050,0.076952,0.120602,0.197554,0.026717
97,IL6_TNFSF18,6248,385,589,523,974,0.061620,0.094270,0.155890,0.083707
28,IL21_TNF,6405,431,519,677,950,0.067291,0.081030,0.148322,0.105699


## Step 4: Combine reinforcement + synergy and export

Left-merges the step-3 synergy summary (`summary`, 138 `consistent_ligands` combos)
with the step-2 reinforcement result (`result`) on `interaction`, so the synergy
summary's 138-combo row count is preserved and each row is annotated with its
reinforcement scores. Writes exactly one output CSV:
`analysis_outs/signal_reinforcement_lfc_mean/results_combined_0.1filter.csv`.


In [16]:
REINF_OUT_DIR = f'{OUTS_DIR}/signal_reinforcement_lfc_mean'
os.makedirs(REINF_OUT_DIR, exist_ok=True)
RESULTS_OUT_CSV = f'{REINF_OUT_DIR}/results_combined_0.1filter.csv'

combined = summary.merge(result, on='interaction', how='left', suffixes=('', '_reinf'))

cols = [
    'interaction', 'ligand1', 'ligand2',
    'lfc_mean_1to2', 'lfc_mean_2to1', 'lfc_mean_sum',
    'sig_flag_1to2', 'sig_flag_2to1', 'sig_flag_final',
    'n_receptor_genes_scored_1to2', 'n_receptor_genes_scored_2to1',
    'receptor_genes_scored_1to2', 'receptor_genes_scored_2to1',
    'has_1to2_data', 'has_2to1_data',
    'uses_shared_subunit_only_1to2', 'uses_shared_subunit_only_2to1',
    'n_total_degs', 'n_synergy_positive', 'n_synergy_negative', 'n_synergy_total', 'n_buffering',
    'freq_synergy_positive', 'freq_synergy_negative', 'freq_synergy_total', 'freq_buffering',
]
combined = combined[cols]
assert len(combined) == len(summary), 'left-merge must preserve the 138 consistent_ligands row count'
combined.to_csv(RESULTS_OUT_CSV, index=False)
print('wrote', RESULTS_OUT_CSV, combined.shape)
print('sig_flag_final True:', int(combined['sig_flag_final'].sum()), '/', len(combined))
combined.head()


wrote /data1/rudenska/EYW/git_projects/SIG13/analysis_outs/signal_reinforcement_lfc_mean/results_combined_0.1filter.csv (138, 26)
sig_flag_final True: 50 / 138


,interaction,ligand1,ligand2,lfc_mean_1to2,lfc_mean_2to1,lfc_mean_sum,sig_flag_1to2,sig_flag_2to1,sig_flag_final,n_receptor_genes_scored_1to2,...,uses_shared_subunit_only_2to1,n_total_degs,n_synergy_positive,n_synergy_negative,n_synergy_total,n_buffering,freq_synergy_positive,freq_synergy_negative,freq_synergy_total,freq_buffering
0,IFNG_CCL19,IFNG,CCL19,-0.140660,-0.134862,-0.275521,False,False,False,1.0,...,False,1839,7,8,15,59,0.003806,0.004350,0.008157,0.032083
1,IFNG_CCL21A,IFNG,CCL21A,-0.141371,-0.295868,-0.437240,False,False,False,1.0,...,False,2003,7,7,14,24,0.003495,0.003495,0.006990,0.011982
2,IFNG_IFNA,IFNG,IFNA,0.017523,-0.593968,-0.576445,False,False,False,2.0,...,False,3356,26,32,58,303,0.007747,0.009535,0.017282,0.090286
3,IFNG_IFNB1,IFNG,IFNB1,0.013164,-0.944274,-0.931110,False,False,False,2.0,...,False,2971,22,22,44,271,0.007405,0.007405,0.014810,0.091215
4,IFNG_IFNK,IFNG,IFNK,0.014273,-0.450799,-0.436526,False,False,False,2.0,...,False,2856,20,21,41,216,0.007003,0.007353,0.014356,0.075630
